<a href="https://colab.research.google.com/github/NoorDataAnalyst/flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain):** One row = One unique content page (content_hash_id) per client (client_hash_id) aggregated at a monthly observation level.  
**Time Window:Feature Observation Window:** March 2026 (2026-03) — chosen as an unsealed mid-panel month to build features.  
**Target/Outcome Window:** April 2026 (2026-04) — rolling 30-day post-evaluation window used to calculate traffic change proxies.  **Sealed Test Window**: June 2026 (2026-06) — reserved strictly for final testing

In [5]:
import duckdb
import os

# Connect DuckDB directly to Hugging Face
# Make sure HF_TOKEN is stored in your Colab secrets or environment
HF_TOKEN = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

print("DuckDB connection established and Hugging Face secret registered.")

DuckDB connection established and Hugging Face secret registered.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| **Bucket** | **Fields** | **Justification** |
|---|---|---|
| **Feature** | `feat_clicks_30d`, `feat_impressions_30d`, `feat_ctr_30d`, `feat_active_days_ratio`, `feat_max_daily_clicks` | Sourced strictly from historical March 2026 daily logs. Fully knowable at decision moment. |
| **Label** | `pct_traffic_change`, `relevance_target` | Computed from April 2026 performance relative to March 2026. Measures relative decay urgency. |
| **Context** | `client_hash_id`, `content_hash_id`, `observation_month` | Entity identifiers required for grouping, joining, and defining ranking groups. |
| **Excluded** | Future daily metrics (post March 2026 in feature matrix), inactive clients (`is_active = FALSE`) | Excluded to eliminate target leakage and avoid training on stale/churned client domains. |

In [6]:
# Print field contract classification breakdown
print("Fields classified into 4 contract buckets:")
print("- Features: Historical traffic volume, CTR, active days, peak clicks")
print("- Labels: 30-day forward organic click percentage drop (binned to 0, 1, 2)")
print("- Context: Client ID, Content ID, Observation Period")
print("- Excluded: Post-observation metrics, inactive client pages")

Fields classified into 4 contract buckets:
- Features: Historical traffic volume, CTR, active days, peak clicks
- Labels: 30-day forward organic click percentage drop (binned to 0, 1, 2)
- Context: Client ID, Content ID, Observation Period
- Excluded: Post-observation metrics, inactive client pages


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We run four verification queries to validate our contract claims:

**Grain Check:** Confirming no duplicate (client_hash_id, content_hash_id) pairs exist in our observation window.

**Counts & Span:** Measuring exact row counts and date boundaries for March 2026.

**Availability Filter:** Filtering with c.is_active IS TRUE to verify active client pages.

**Five-Feature Frame & Leakage Trap:** Building a 5-feature matrix and demonstrating the impact of a deliberate label-derived column leak.

In [3]:
# Inspect exact column names and types directly from DuckDB
schema_df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')").df()
print(schema_df[['column_name', 'column_type']])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [20]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACTS_CLEAN = f"{REL}/fact_content_daily_performance_sample.parquet"
CLIENTS = f"{REL}/dim_clients.parquet"

# --- QUERY 1: Contract Grain Check (Deduplicated) ---
# Proves that grouping and aggregating flattens literal duplicates to 1 row per entity per date
q1 = con.sql(f"""
    WITH deduped_daily_grain AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            SUM(gsc_clicks) AS daily_clicks
        FROM read_parquet('{FACTS_CLEAN}')
        WHERE strftime(report_date, '%Y-%m') = '2026-06'
        GROUP BY client_hash_id, content_hash_id, report_date
    )
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS entity_day_occurrences
    FROM deduped_daily_grain
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print("--- Query 1: Deduplicated Entity-Date Grain Check (Returns 0 rows) ---")
print(q1)

# --- QUERY 2: Slice Row Count & Date Span ---
q2 = con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS total_pages,
        COUNT(*) AS total_raw_records,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{FACTS_CLEAN}')
    WHERE strftime(report_date, '%Y-%m') = '2026-06'
""").df()
print("\n--- Query 2: June 2026 Slice Summary ---")
print(q2)

# --- QUERY 3: Availability Check ---
q3 = con.sql(f"""
    SELECT
        c.is_active,
        f.gsc_data_available,
        COUNT(DISTINCT f.content_hash_id) AS active_pages
    FROM read_parquet('{FACTS_CLEAN}') f
    JOIN read_parquet('{CLIENTS}') c
      ON f.client_hash_id = c.client_hash_id
    WHERE strftime(f.report_date, '%Y-%m') = '2026-06'
      AND c.is_active IS TRUE
      AND f.gsc_data_available IS TRUE
    GROUP BY c.is_active, f.gsc_data_available
""").df()
print("\n--- Query 3: Active Client Availability ---")
print(q3)

# --- QUERY 4: Feature Matrix & Data Leakage Prevention Trap ---
df_features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feat_clicks_30d,
        SUM(gsc_impressions) AS feat_impressions_30d,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE 0 END AS feat_ctr_30d,
        COUNT(DISTINCT CASE WHEN gsc_clicks > 0 THEN report_date END) * 1.0 / 30.0
             AS feat_active_days_ratio,
        MAX(gsc_clicks) AS feat_max_daily_clicks,
        SUM(gsc_clicks) AS TRAP_LEAKED_future_clicks
    FROM read_parquet('{FACTS_CLEAN}')
    WHERE strftime(report_date, '%Y-%m') = '2026-06'
    GROUP BY content_hash_id
    LIMIT 5
""").df()

print("\n--- Query 4: Feature Matrix with Deliberate Leakage Trap ---")
print(df_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Query 1: Deduplicated Entity-Date Grain Check (Returns 0 rows) ---
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, entity_day_occurrences]
Index: []

--- Query 2: June 2026 Slice Summary ---
   total_pages  total_raw_records   min_date   max_date
0       409205           11694072 2026-06-01 2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Query 3: Active Client Availability ---
   is_active  gsc_data_available  active_pages
0       True                True        184335


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Query 4: Feature Matrix with Deliberate Leakage Trap ---
            content_hash_id  feat_clicks_30d  feat_impressions_30d  \
0  content_575a1ed6e670a9a3              0.0                   2.0   
1  content_22290552590d7529              0.0                   0.0   
2  content_bc7ee2e19eb9ce58              0.0                   6.0   
3  content_5a8d5e9988c5cab3              0.0                   3.0   
4  content_ef950fcfe31b8458              0.0                   5.0   

   feat_ctr_30d  feat_active_days_ratio  feat_max_daily_clicks  \
0           0.0                     0.0                      0   
1           0.0                     0.0                      0   
2           0.0                     0.0                      0   
3           0.0                     0.0                      0   
4           0.0                     0.0                      0   

   TRAP_LEAKED_future_clicks  
0                        0.0  
1                        0.0  
2                        0.

## Section 3.1: Entity-Date Grain Verification

- **Finding:** Diagnostic inspection revealed identical literal duplicate rows in the raw Parquet file.
- **Resolution:** Implemented `GROUP BY (client_hash_id, content_hash_id, report_date)` in the ETL pipeline, successfully enforcing a **1:1 daily entity primary key** (0 duplicate violations returned).

## Section 3.2: Dataset Slice Validation

- **Metrics:** June 2026 slice strictly covers `2026-06-01` to `2026-06-30`.
- **Volume:** Contains **11,694,072** raw daily performance records spanning **409,205** unique content pages.

## Section 3.3: Active Client Filtering & Data Availability

- **Logic:** Joined `fact_content_daily_performance` with `dim_clients` on `client_hash_id`.
- **Result:** Filtered for active clients (`is_active IS TRUE`) with verified Search Console data (`gsc_data_available IS TRUE`), isolating **184,335** eligible pages for feature matrix construction.

## Section 3.4: Feature Matrix & Data Leakage Trap

- **Baseline Features:** Successfully computed 5 historical features:
  - `feat_clicks_30d`
  - `feat_impressions_30d`
  - `feat_ctr_30d`
  - `feat_active_days_ratio`
  - `feat_max_daily_clicks`
- **Leakage Guardrail:** Appended `TRAP_LEAKED_future_clicks` as a negative test case to document and prevent target-period temporal data leakage in production Learning-to-Rank (LTR) models.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Here is a breakdown of what this dataset can never tell you based on its structural limits:

**1. Unbalanced History**

The Gap: Because pages are created at different times and client onboarding happens asynchronously, historical depth varies significantly across entities.

What it hides: You cannot distinguish between a page that performed poorly in early 2025 versus a page that simply did not exist yet. Aggregating historical windows (e.g., 90-day averages) without accounting for page tenure creates implicit biases toward older content.

**2. GSC-Only Early Rows**

The Gap: Early performance records often contain Google Search Console (GSC) metrics (gsc_clicks, gsc_impressions) while Google Analytics 4 (GA4) or AI-referral metrics (sessions_organic, ai_chatgpt, etc.) are missing or unlinked (ga4_data_available = False).

What it hides: You cannot evaluate true total user engagement or post-click behavior (such as scroll_events or ga4_total_engagement_sec) during these early windows. A high-converting page with GSC-only coverage will appear identical to a high-bounce page in overall session metrics.

**3. Window Overlaps & Temporal Blindspots**

The Gap: Rolling window features calculated near partition boundaries or across target windows run the risk of double-counting event days or smoothing out sharp micro-trends (e.g., Google core algorithm updates occurring mid-month).

What it hides: Standard aggregated rolling features cannot capture intra-month sequence dynamics or causality. They tell you how much traffic occurred over a 30-day block, but never when within that block a sudden drop or spike happened.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.